In [1]:
import subprocess, sys, os, getpass

def _install():
    pkgs = [
        "langchain", "langchain-community", "langchain-groq", "langchain-core",
        "langchain-text-splitters", "sentence-transformers", "chromadb",
        "rank_bm25", "tavily-python", "wikipedia", "pypdf", "gradio>=5.0,<6",
    ]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print(" Installing dependencies (silent)...")
_install()
print(" Dependencies ready.")

import re, ast, json, time, operator as op, warnings
from datetime import datetime
warnings.filterwarnings("ignore")

def get_key(name):
    val = None
    try:
        from google.colab import userdata
        val = userdata.get(name)
    except Exception:
        pass
    val = val or os.environ.get(name)
    if not val:
        val = getpass.getpass(f"🔑 Enter {name} (leave blank to skip): ").strip()
    if val:
        os.environ[name] = val
    return val

GROQ_API_KEY = get_key("GROQ_API_KEY")
TAVILY_API_KEY = get_key("TAVILY_API_KEY")

import torch
import numpy as np
import gradio as gr
import chromadb
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from langchain_core.tools import tool
from langchain_core.messages import (SystemMessage, HumanMessage, AIMessage,
                                     ToolMessage)
from langchain_groq import ChatGroq
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_text_splitters import RecursiveCharacterTextSplitter

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f" Device: {DEVICE}")

import inspect
print(f"Gradio version: {gr.__version__}")

def _kw(fn, **kwargs):
    """Keep only kwargs the installed Gradio version accepts (works on Gradio 5 and 6)."""
    params = inspect.signature(fn).parameters
    if any(p.kind == inspect.Parameter.VAR_KEYWORD for p in params.values()):
        # **kwargs present: still filter known-removed args to be safe
        return {k: v for k, v in kwargs.items() if k in params or k not in ("type", "show_copy_button", "css", "theme", "js")}
    return {k: v for k, v in kwargs.items() if k in params}

def _text(content):
    """Extract plain text from str or Gradio-6 style content blocks."""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return "".join(b.get("text", "") if isinstance(b, dict) else str(b) for b in content)
    return ""

# MODEL ROUTER
FAST_MODEL = "openai/gpt-oss-20b"
DEEP_MODEL = "openai/gpt-oss-120b"

DEEP_KEYWORDS = [
    "analyze", "analyse", "compare", "contrast", "research", "synthesize",
    "synthesise", "in depth", "in-depth", "deep dive", "comprehensive",
    "evaluate", "pros and cons", "trade-off", "tradeoff", "step by step",
    "explain why", "implications", "summarize the document", "knowledge base",
    "uploaded", "document", "multi-hop", "relationship between", "history of",
]
MATH_PATTERN = re.compile(r"^[\s\d\.\+\-\*/\^\(\)%=x×÷]+$|\b(calculate|compute|add|multiply|sum of|product of)\b", re.I)

def route(text: str):
    """Lightweight heuristic intent classifier -> (model_id, reason)."""
    t = text.lower()
    words = len(t.split())
    score = 0
    reasons = []
    hits = [k for k in DEEP_KEYWORDS if k in t]
    if hits:
        score += 2; reasons.append(f"keywords:{hits[0]}")
    if words > 45:
        score += 1; reasons.append("long query")
    if t.count("?") > 1 or " and then " in t or " also " in t:
        score += 1; reasons.append("multi-part")
    if MATH_PATTERN.search(t) and words < 25:
        score -= 2; reasons.append("math")
    if score >= 2:
        return DEEP_MODEL, "deep tier (" + ", ".join(reasons) + ")"
    return FAST_MODEL, "fast tier" + (f" ({', '.join(reasons)})" if reasons else "")

def make_llm(model: str, temperature: float):
    return ChatGroq(model=model, temperature=temperature, streaming=True)

#TOOLS + ATTRIBUTION
SOURCE_TAGS = []   # (key, formatted tag) collected per request
TOOL_LOG = []      # live tool activity log

def log(msg: str):
    TOOL_LOG.append(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")

def log_text():
    return "\n".join(TOOL_LOG[-14:]) if TOOL_LOG else "No tool activity yet."

# --- Tavily web search ---
_tavily_tool = None
try:
    from langchain_community.tools.tavily_search import TavilySearchResults
    if TAVILY_API_KEY:
        _tavily_tool = TavilySearchResults(max_results=3)
except Exception:
    _tavily_tool = None

def _tavily_raw(query: str):
    """Returns list of {title,url,content}. Uses TavilySearchResults, falls back to tavily-python."""
    if _tavily_tool is not None:
        try:
            res = _tavily_tool.invoke(query)
            if isinstance(res, list):
                return [{"title": r.get("title") or r.get("url", ""), "url": r.get("url", ""),
                         "content": r.get("content", "")} for r in res]
        except Exception as e:
            log(f"Tavily community tool failed ({str(e)[:60]}); using client")
    from tavily import TavilyClient
    client = TavilyClient(api_key=TAVILY_API_KEY)
    res = client.search(query, max_results=3)
    return [{"title": r.get("title", ""), "url": r.get("url", ""), "content": r.get("content", "")}
            for r in res.get("results", [])]

@tool
def web_search(query: str) -> str:
    """Search the live web for current events, news, recent facts and breaking updates."""
    if not TAVILY_API_KEY:
        return "ERROR: TAVILY_API_KEY not configured; TAVILY unavailable."
    try:
        results = _tavily_raw(query)
        if not results:
            return "No web results found."
        out = []
        for r in results:
            label = r["title"] or r["url"]
            tag = f">  **Source:** Tavily Web Search ({label} — {r['url']})"
            SOURCE_TAGS.append((r["url"], tag))
            out.append(f"{r['content'][:700]}\n{tag}")
        return "\n\n".join(out)
    except Exception as e:
        return f"ERROR: TAVILY SEARCH failed ({str(e)[:150]})."

_wiki = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(top_k_results=2, doc_content_chars_max=1500))

@tool
def wikipedia(query: str) -> str:
    """Look up historical, biographical, scientific or encyclopedic facts on Wikipedia."""
    try:
        text = _wiki.invoke(query)
        titles = re.findall(r"Page:\s*(.+)", text)
        if not titles or "No good Wikipedia Search Result" in text:
            return "No Wikipedia article found."
        tags = []
        for t in titles:
            tag = f">  **Source:** Wikipedia ({t.strip()})"
            SOURCE_TAGS.append((t.strip(), tag)); tags.append(tag)
        return text + "\n" + "\n".join(tags)
    except Exception as e:
        return f"ERROR: Wikipedia lookup failed ({str(e)[:150]})."

_OPS = {ast.Add: op.add, ast.Sub: op.sub, ast.Mult: op.mul, ast.Div: op.truediv,
        ast.Pow: op.pow, ast.Mod: op.mod, ast.FloorDiv: op.floordiv,
        ast.USub: op.neg, ast.UAdd: op.pos}

def _safe_eval(node):
    if isinstance(node, ast.Expression):
        return _safe_eval(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)) and not isinstance(node.value, bool):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        l, r = _safe_eval(node.left), _safe_eval(node.right)
        if isinstance(node.op, ast.Pow) and abs(r) > 1000:
            raise ValueError("Exponent too large")
        return _OPS[type(node.op)](l, r)
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_safe_eval(node.operand))
    raise ValueError("Unsupported expression")

def _math_tag(expr: str, result) -> str:
    tag = f">  **Verified via Math Tool:** {expr} = {result}"
    SOURCE_TAGS.append((expr, tag))
    return tag

@tool
def add(a: float, b: float) -> str:
    """Add two numbers exactly."""
    r = a + b
    return f"{a} + {b} = {r}\n{_math_tag(f'{a} + {b}', r)}"

@tool
def multiply(a: float, b: float) -> str:
    """Multiply two numbers exactly."""
    r = a * b
    return f"{a} * {b} = {r}\n{_math_tag(f'{a} * {b}', r)}"

@tool
def calculate(expression: str) -> str:
    """Safely evaluate an arithmetic expression (+ - * / // % ** and parentheses), e.g. '(12.5*4)/3'."""
    try:
        expr = expression.replace("^", "**").replace("×", "*").replace("÷", "/")
        r = _safe_eval(ast.parse(expr.strip(), mode="eval"))
        return f"{expression} = {r}\n{_math_tag(expression, r)}"
    except Exception as e:
        return f"ERROR: could not evaluate '{expression}' ({e})."

# TWO-STAGE RAG
print(" Loading embedding + cross-encoder models...")
EMBEDDER = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=DEVICE)
RERANKER = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device=DEVICE)
print(" RAG models loaded.")

_chroma = chromadb.Client()
KB = {"collection": None, "chunks": [], "meta": [], "ids": [], "bm25": None}

def _new_collection():
    try:
        _chroma.delete_collection("knowledge_base")
    except Exception:
        pass
    KB["collection"] = _chroma.create_collection("knowledge_base", metadata={"hnsw:space": "cosine"})
    KB.update(chunks=[], meta=[], ids=[], bm25=None)

_new_collection()
_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)

def _tok(s: str):
    return re.findall(r"\w+", s.lower())

def _read_file(path: str) -> str:
    if path.lower().endswith(".pdf"):
        from pypdf import PdfReader
        return "\n".join((p.extract_text() or "") for p in PdfReader(path).pages)
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

def ingest(files, pasted_text):
    added = 0
    try:
        sources = []
        for f in (files or []):
            p = f if isinstance(f, str) else getattr(f, "name", str(f))
            sources.append((os.path.basename(p), _read_file(p)))
        if pasted_text and pasted_text.strip():
            sources.append(("pasted_text", pasted_text))
        if not sources:
            return " Nothing to ingest. Upload files or paste text.\n" + kb_stats()
        for name, text in sources:
            chunks = [c for c in _splitter.split_text(text) if c.strip()]
            if not chunks:
                continue
            start = len(KB["chunks"])
            ids = [f"chunk-{start + i}" for i in range(len(chunks))]
            embs = EMBEDDER.encode(chunks, batch_size=64, normalize_embeddings=True,
                                   show_progress_bar=False).tolist()
            KB["collection"].add(ids=ids, documents=chunks, embeddings=embs,
                                 metadatas=[{"source": name}] * len(chunks))
            KB["chunks"] += chunks; KB["ids"] += ids; KB["meta"] += [name] * len(chunks)
            added += len(chunks)
        if KB["chunks"]:
            KB["bm25"] = BM25Okapi([_tok(c) for c in KB["chunks"]])
        log(f" Ingested {added} chunks into knowledge base")
        return f" Ingested {added} new chunks.\n" + kb_stats()
    except Exception as e:
        return f" Ingestion failed: {str(e)[:200]}\n" + kb_stats()

def clear_kb():
    _new_collection()
    log(" Knowledge base cleared")
    return " Knowledge base cleared.\n" + kb_stats()

def kb_stats():
    return f"KB: {len(KB['chunks'])} chunks from {len(set(KB['meta']))} source(s)"

def retrieve(query: str, top_k: int = 4, pool: int = 12):
    n = len(KB["chunks"])
    if n == 0:
        return []
    # Stage 1: hybrid dense (Chroma) + sparse (BM25) via Reciprocal Rank Fusion
    q_emb = EMBEDDER.encode([query], normalize_embeddings=True, show_progress_bar=False).tolist()
    dense = KB["collection"].query(query_embeddings=q_emb, n_results=min(15, n))["ids"][0]
    scores = KB["bm25"].get_scores(_tok(query))
    sparse_idx = [i for i in np.argsort(scores)[::-1][:15] if scores[i] > 0]
    sparse = [KB["ids"][i] for i in sparse_idx]
    fused = {}
    for ranking in (dense, sparse):
        for rank, cid in enumerate(ranking):
            fused[cid] = fused.get(cid, 0.0) + 1.0 / (60 + rank)
    cand_ids = [c for c, _ in sorted(fused.items(), key=lambda x: -x[1])[:pool]]
    # Stage 2: cross-encoder re-ranking on GPU
    id_to_i = {cid: i for i, cid in enumerate(KB["ids"])}
    pairs = [(query, KB["chunks"][id_to_i[c]]) for c in cand_ids]
    ce = RERANKER.predict(pairs)
    ranked = sorted(zip(cand_ids, ce), key=lambda x: -x[1])[:top_k]
    return [(cid, float(s), KB["chunks"][id_to_i[cid]], KB["meta"][id_to_i[cid]]) for cid, s in ranked]

@tool
def search_knowledge_base(query: str) -> str:
    """Search the user's ingested local documents (2-stage hybrid RAG). Use for questions about uploaded documents."""
    try:
        hits = retrieve(query)
        if not hits:
            return "Knowledge base is empty or no relevant chunks were found."
        out = []
        for cid, score, text, src in hits:
            tag = f">  **Source:** Ingested Knowledge Base ({cid} · {src} · score {score:.2f})"
            SOURCE_TAGS.append((cid, tag))
            out.append(f"{text}\n{tag}")
        return "\n\n".join(out)
    except Exception as e:
        return f"ERROR: knowledge base search failed ({str(e)[:150]})."

TOOLS = [web_search, wikipedia, add, multiply, calculate, search_knowledge_base]
TOOL_MAP = {t.name: t for t in TOOLS}

def run_tool(name: str, args: dict) -> str:
    t0 = time.time()
    try:
        if name not in TOOL_MAP:
            return f"ERROR: unknown tool '{name}'."
        res = str(TOOL_MAP[name].invoke(args))
    except Exception as e:
        res = f"ERROR: tool '{name}' failed ({str(e)[:150]})."
    log(f"🔧 {name}({json.dumps(args)[:70]}) -> {len(res)} chars in {time.time() - t0:.1f}s")
    return res

#SYSTEM PROMPT
SYSTEM_PROMPT = """You are an autonomous research and problem-solving agent.

STYLE
- Be direct, comprehensive and neutral. Answer the question asked, plainly.
- Skip moralizing, lectures, and generic boilerplate disclaimers. Only add a caveat when it materially changes the answer.

GROUNDED VERIFICATION (zero tolerance for hallucination)
- For real-world facts, dates, news, people or numbers: call the appropriate tool FIRST, read the result, then answer strictly from the retrieved evidence.
- web_search -> current events/breaking news. wikipedia -> history, biography, science.
- ALL arithmetic must go through add / multiply / calculate. Never compute in your head.
- search_knowledge_base -> anything about the user's uploaded documents.
- If tools return nothing useful, say so instead of guessing.

MANDATORY SOURCE ATTRIBUTION
- Tool outputs end with ready-made source lines . Copy the relevant ones verbatim, on their own lines, in your final answer, right after the claims they support.
- Calculations must show the verification line: >  **Verified via Math Tool:** <calculation>
- Use markdown; put code in fenced blocks with a language tag."""

#AGENT LOOP (STREAMING)
MAX_STEPS = 6
MODEL_CHOICES = ["Auto (Router)", FAST_MODEL, DEEP_MODEL]

def to_lc(history):
    out = []
    for m in history:
        if m.get("metadata"):
            continue  # skip tool drawers
        c = _text(m.get("content"))
        if not c.strip():
            continue
        out.append(HumanMessage(c) if m["role"] == "user" else AIMessage(c))
    return out

API_STATUS = {"ok": None}

def check_api():
    try:
        ChatGroq(model=FAST_MODEL, max_tokens=5).invoke("ping")
        API_STATUS["ok"] = True
    except Exception as e:
        API_STATUS["ok"] = False
        print(" Groq API check failed:", str(e)[:150])

def header_html(model="—", state="idle", ttft=None, total=None):
    ok = API_STATUS["ok"]
    dot = "#49CC90" if ok else ("#F93E3E" if ok is False else "#FCA130")
    api = "API Connected" if ok else ("API Error" if ok is False else "API Unchecked")
    lat = "—" if ttft is None else f"TTFT {ttft*1000:.0f} ms"
    if total is not None:
        lat += f" · total {total:.1f}s"
    pct = 0 if ttft is None else max(5, min(100, int(100 - ttft * 60)))
    return f"""
    <div class="pm-nav">
      <div class="pm-left"><span class="pm-logo">◉</span><b>VAULT</b>
        <span class="pm-crumb">/ Groq Collection / <span class="pm-method">POST</span> /v1/chat/completions</span></div>
      <div class="pm-right">
        <span class="pm-pill"><span class="pm-dot" style="background:{dot}"></span>{api}</span>
        <span class="pm-pill"> {lat}<span class="pm-gauge"><span style="width:{pct}%"></span></span></span>
        <span class="pm-badge">{model}</span>
        <span class="pm-pill">{state}</span>
      </div>
    </div>"""

def respond(message, history, model_choice, temperature):
    message = (message or "").strip()
    history = list(history or [])
    if not message:
        yield "", history, log_text(), header_html()
        return

    history.append({"role": "user", "content": message})
    SOURCE_TAGS.clear()
    t0 = time.time()
    if model_choice == "Auto (Router)":
        model, why = route(message)
    else:
        model, why = model_choice, "manual override"
    log(f" Router -> {model} [{why}]")
    lc = [SystemMessage(SYSTEM_PROMPT)] + to_lc(history[:-1]) + [HumanMessage(message)]
    ttft, fell_back = None, False
    yield "", history, log_text(), header_html(model, "thinking…")

    for step in range(MAX_STEPS):
        llm = make_llm(model, temperature).bind_tools(TOOLS)
        full, cur = None, None
        try:
            for chunk in llm.stream(lc):
                full = chunk if full is None else full + chunk
                text = chunk.content if isinstance(chunk.content, str) else ""
                if text:
                    if ttft is None:
                        ttft = time.time() - t0
                    if cur is None:
                        cur = {"role": "assistant", "content": ""}
                        history.append(cur)
                    cur["content"] += text
                    yield "", history, log_text(), header_html(model, "streaming…", ttft)
        except Exception as e:
            log(f" {model} error: {str(e)[:110]}")
            if cur is not None and cur in history:
                history.remove(cur)
            if model == FAST_MODEL and not fell_back:
                fell_back, model = True, DEEP_MODEL
                log("↪ Falling back to deep-tier model")
                continue
            history.append({"role": "assistant", "content": f" Request failed: `{str(e)[:250]}`"})
            break

        if full is None or not getattr(full, "tool_calls", None):
            break

        lc.append(full)
        for tc in full.tool_calls:
            name, args = tc["name"], tc.get("args", {})
            drawer = {"role": "assistant",
                      "content": f"**Arguments**\n```json\n{json.dumps(args, indent=2)}\n```\n⏳ running…",
                      "metadata": {"title": f" Tool call: {name}", "status": "pending"}}
            history.append(drawer)
            yield "", history, log_text(), header_html(model, f"tool: {name}", ttft)
            result = run_tool(name, args)
            drawer["content"] = (f"**Arguments**\n```json\n{json.dumps(args, indent=2)}\n```\n"
                                 f"**Result**\n```\n{result[:1500]}\n```")
            drawer["metadata"]["status"] = "done"
            lc.append(ToolMessage(content=result, tool_call_id=tc["id"], name=name))
            yield "", history, log_text(), header_html(model, f"tool: {name} ✓", ttft)

    # Guarantee attribution footer for any source the model forgot to cite
    answer_text = "\n".join(_text(m["content"]) for m in history
                            if m["role"] == "assistant" and not m.get("metadata"))
    seen, missing = set(), []
    for key, tag in SOURCE_TAGS:
        if key and key not in answer_text and tag not in seen:
            seen.add(tag); missing.append(tag)
    if missing:
        footer = "\n\n---\n**Sources used**\n\n" + "\n\n".join(missing)
        last = next((m for m in reversed(history)
                     if m["role"] == "assistant" and not m.get("metadata")), None)
        if last is not None and isinstance(last["content"], str) and last["content"]:
            last["content"] += footer
        else:
            history.append({"role": "assistant", "content": footer.strip()})

    log(f" Done in {time.time() - t0:.1f}s")
    yield "", history, log_text(), header_html(model, "done ✓", ttft, time.time() - t0)

# ----------------------------- 7. ui --------------------------
CSS = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600&family=JetBrains+Mono:wght@400;500&display=swap');
:root, .dark, .gradio-container {
  --body-background-fill:#1C1C1C; --background-fill-primary:#212121; --background-fill-secondary:#1C1C1C;
  --block-background-fill:#212121; --block-border-color:#383838; --border-color-primary:#383838;
  --body-text-color:#E6E6E6; --block-label-text-color:#9B9B9B; --block-title-text-color:#E6E6E6;
  --input-background-fill:#1C1C1C; --input-border-color:#383838; --input-border-color-focus:#FF6C37;
  --button-primary-background-fill:#FF6C37; --button-primary-background-fill-hover:#E85A25;
  --button-primary-text-color:#fff; --color-accent:#FF6C37; --slider-color:#FF6C37;
  --button-secondary-background-fill:#2B2B2B; --button-secondary-text-color:#E6E6E6;
  --button-secondary-border-color:#383838; --shadow-drop:none;
}
body, .gradio-container { background:#1C1C1C !important; font-family:'Inter',sans-serif !important; }
footer { display:none !important; }
.pm-nav { display:flex; justify-content:space-between; align-items:center; gap:12px; flex-wrap:wrap;
  background:#212121; border:1px solid #383838; border-radius:8px; padding:10px 16px; }
.pm-left { display:flex; align-items:center; gap:10px; color:#E6E6E6; font-size:14px; }
.pm-logo { color:#FF6C37; font-size:20px; }
.pm-crumb { color:#8A8A8A; font-family:'JetBrains Mono',monospace; font-size:12px; }
.pm-method { color:#49CC90; font-weight:600; }
.pm-right { display:flex; align-items:center; gap:8px; flex-wrap:wrap; }
.pm-pill { background:#2B2B2B; border:1px solid #383838; border-radius:14px; padding:4px 10px;
  font-family:'JetBrains Mono',monospace; font-size:11px; color:#CFCFCF; display:inline-flex; align-items:center; gap:6px; }
.pm-dot { width:8px; height:8px; border-radius:50%; display:inline-block; }
.pm-gauge { width:46px; height:5px; background:#383838; border-radius:3px; overflow:hidden; display:inline-block; }
.pm-gauge span { display:block; height:100%; background:#FF6C37; }
.pm-badge { background:#FF6C37; color:#fff; border-radius:4px; padding:4px 10px;
  font-family:'JetBrains Mono',monospace; font-size:11px; font-weight:500; }
.side { background:#212121; border:1px solid #383838; border-radius:8px; padding:12px !important; }
.side-title { color:#FF6C37; font-family:'JetBrains Mono',monospace; font-size:12px; letter-spacing:1px; margin:4px 0 8px; }
#chatbot { border:1px solid #383838 !important; border-radius:8px; background:#1C1C1C !important; }
#chatbot .message, #chatbot .bubble-wrap { font-family:'Inter',sans-serif; }
#chatbot pre, #chatbot code { font-family:'JetBrains Mono',monospace !important; background:#161616 !important; }
#chatbot blockquote { border-left:3px solid #FF6C37; background:#262626; padding:4px 10px; margin:6px 0; color:#CFCFCF; }
#toollog textarea { font-family:'JetBrains Mono',monospace !important; font-size:11px !important; color:#49CC90 !important; background:#161616 !important; }
"""

FORCE_DARK_JS = "() => { document.body.classList.add('dark'); }"

THEME = gr.themes.Base(primary_hue="orange", neutral_hue="slate")
_BLOCKS_STYLE = _kw(gr.Blocks.__init__, css=CSS, theme=THEME, js=FORCE_DARK_JS)
_LAUNCH_STYLE = {} if len(_BLOCKS_STYLE) == 3 else _kw(gr.Blocks.launch, css=CSS, theme=THEME, js=FORCE_DARK_JS)
_BLOCKS_STYLE = _BLOCKS_STYLE if len(_BLOCKS_STYLE) == 3 else {}

with gr.Blocks(title="Groq VAULT", **_BLOCKS_STYLE) as demo:

    header = gr.HTML(header_html())

    with gr.Row(equal_height=False):
        with gr.Column(scale=3, elem_classes="side"):
            gr.HTML('<div class="side-title">▸ CONTROL PANEL</div>')
            model_dd = gr.Dropdown(MODEL_CHOICES, value="Auto (Router)", label="Model")
            temp = gr.Slider(0.0, 1.5, value=0.3, step=0.05, label="Temperature")
            gr.HTML('<div class="side-title">▸ TOOL ACTIVITY</div>')
            tool_log = gr.Textbox(value=log_text(), lines=9, max_lines=9, interactive=False,
                                  show_label=False, elem_id="toollog")
            gr.HTML('<div class="side-title">▸ KNOWLEDGE BASE (2-STAGE RAG)</div>')
            files = gr.File(label="Upload .txt / .md / .pdf", file_count="multiple",
                            file_types=[".txt", ".md", ".pdf", ".csv"], type="filepath")
            paste = gr.Textbox(label="…or paste text", lines=3, placeholder="Paste reference text here")
            with gr.Row():
                ingest_btn = gr.Button("Ingest", variant="primary")
                clear_btn = gr.Button("Clear KB")
            ingest_status = gr.Textbox(value=kb_stats(), show_label=False, interactive=False, lines=2)

        with gr.Column(scale=8):
            chatbot = gr.Chatbot(**_kw(gr.Chatbot, type="messages", height=580, elem_id="chatbot",
                                 show_copy_button=True, render_markdown=True,
                                 label="Response",
                                 placeholder="Ask anything — the agent will use tools and cite sources."))
            with gr.Row():
                msg = gr.Textbox(placeholder="Send a request…  (Enter to send)", show_label=False,
                                 scale=9, lines=1, max_lines=6, autofocus=True)
                send = gr.Button("Send ▶", variant="primary", scale=1)
            gr.Examples(
                examples=[
                    "What are the latest developments in AI chips this week?",
                    "Who was Ada Lovelace and why is she important?",
                    "Calculate (1234.5 * 67) / 3 + 89 ** 2",
                    "Compare the transformer and RNN architectures in depth.",
                    "Summarize the uploaded document.",
                ], inputs=msg)
            clear_chat = gr.Button("Clear conversation")

    io_out = [msg, chatbot, tool_log, header]
    io_in = [msg, chatbot, model_dd, temp]
    msg.submit(respond, io_in, io_out)
    send.click(respond, io_in, io_out)
    clear_chat.click(lambda: ([], "", header_html()), None, [chatbot, msg, header])
    ingest_btn.click(ingest, [files, paste], ingest_status).then(
        lambda: log_text(), None, tool_log)
    clear_btn.click(clear_kb, None, ingest_status).then(
        lambda: log_text(), None, tool_log)

# ----------------------------- 8. LAUNCH -------------------------------------
check_api()
print(" Launching...")
demo.queue().launch(share=True, debug=True, **_LAUNCH_STYLE)

 Installing dependencies (silent)...
 Dependencies ready.


KeyboardInterrupt: Interrupted by user